# Install and import needed libraries

In [ ]:
%pip install git+https://github.com/paula-rj/StratoPy.git

In [ ]:
%pip install numpy --upgrade 

After run the latter cell, restart session.

In [1]:
from stratopy import cloudsat as cs
from stratopy import core

In [2]:
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

#Geo
import cartopy
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader

In [8]:
def plot_cldclass_geos(layers_data, layer_n):
        """
          Function that plots the tipes of clouds in a given atmosphere layer along CloudSat track.
          
        Parameters:
        -----------
        layers_data : Pandas Dataframe. 
          Pandas Dataframe containig Latitude, Longitude, and the cloud types in a given Layer.
        layer_n : int
          Number of layer thaat you want to plot. 0 - 9.

        Returns:
        -----------
        Plot. Tipo de nube en la capa 9 con órbita proyectada en proyeccion geoestacionaria de GOES16, 
        con mapa de costas de fondo.
        """
        layer_str = 'layer_' + str(layer_n)

        #Generamos geodataframe a partir del pd dataframe de entrada
        geo_df = gpd.GeoDataFrame(layers_data.loc[:,['Longitude','Latitude',layer_str]], 
                                geometry=gpd.points_from_xy(layers_data.Longitude, layers_data.Latitude))
        geo_df.crs = {'init': 'epsg:4326'} # EPSG 4326 corresponds to coordinates in latitude and longitude
        #Reprojecting into GOES16 geostationary projection 
        geodf_GOESproj = geo_df.to_crs("+proj=geos +h=35786023.0 +lon_0=-75.0")
        crs=ccrs.Geostationary(central_longitude=-75.0, satellite_height=35786023.0) #proyeccion geoestacionaria para Goes16
        fig_dims = (10, 10)
        fig, axis = plt.subplots(figsize=fig_dims)
        axis = plt.axes(projection=crs)
        axis.gridlines 
        axis.coastlines(resolution='10m',color='blue') 
        sns.scatterplot(x='Longitude', y='Latitude', data=geodf_GOESproj, hue=layer_str,
                        palette='bright',s=2, transform=ccrs.PlateCarree())
        plt.show()

Open a CloudSat File using **Stratopy** package

In [9]:
csobj = cs.read_hdf("../input/cloudtypes/02_2019CLDCLASS/2019059175734_68384_CS_2B-CLDCLASS_GRANULE_P1_R05_E08_F03.hdf")

Cut the file. In this case, the area of interest is Southamerica

In [11]:
csobj = csobj.cut([-45.,10., -78.,-36.])
csobj = csobj[csobj.layer_0 != 0]
csobj

The only layer we will use is layer 0, so we drop the others.

Plot the cutted file

In [12]:
plot_cldclass_geos(csobj, 0)